In [87]:
import pandas as pd
from io import BytesIO
from urllib.request import urlopen, Request

# User-Agent header to allow downloads
url_2025 = "https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/996cfe8d-fb35-40ce-b569-698d51fc683b/resource/0b6e5c52-e993-46d6-8d74-8602ee224457/download/TTC%20Subway%20Delay%20Data%20since%202025.csv"
req = Request(url_2025, headers={"User-Agent": "Mozilla/5.0"})

# open the URL and read raw data into memory
with urlopen(req) as resp:
    data = resp.read()


print("First bytes:", data[:10])


df_2025 = pd.read_csv(BytesIO(data))
df_2025.info()
df_2025
print("2025 loaded:", df_2025.shape)
print(df_2025.head())

First bytes: b'_id,Date,T'
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28191 entries, 0 to 28190
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   _id        28191 non-null  int64 
 1   Date       28191 non-null  object
 2   Time       28191 non-null  object
 3   Day        28191 non-null  object
 4   Station    28191 non-null  object
 5   Code       28191 non-null  object
 6   Min Delay  28191 non-null  int64 
 7   Min Gap    28191 non-null  int64 
 8   Bound      17961 non-null  object
 9   Line       28116 non-null  object
 10  Vehicle    28191 non-null  int64 
dtypes: int64(4), object(7)
memory usage: 2.4+ MB
2025 loaded: (28191, 11)
   _id        Date   Time        Day            Station   Code  Min Delay  \
0    1  2025-01-01  02:10  Wednesday   BATHURST STATION  MUSAN          5   
1    2  2025-01-01  02:30  Wednesday     DUNDAS STATION  MUIRS          0   
2    3  2025-01-01  02:32  Wednesday  BROADVIEW STA

In [88]:

# Map each year to its Excel file URL in dictionary
xlsx_urls = {
    2024: "https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/996cfe8d-fb35-40ce-b569-698d51fc683b/resource/2ee1a65c-da06-4ad1-bdfb-b1a57701e46a/download/ttc-subway-delay-data-2024.xlsx",
    2023: "https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/996cfe8d-fb35-40ce-b569-698d51fc683b/resource/2fbec48b-33d9-4897-a572-96c9f002d66a/download/ttc-subway-delay-data-2023.xlsx",
    2022: "https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/996cfe8d-fb35-40ce-b569-698d51fc683b/resource/441143ca-8194-44ce-a954-19f8141817c7/download/ttc-subway-delay-data-2022.xlsx",
    2021: "https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/996cfe8d-fb35-40ce-b569-698d51fc683b/resource/c6e4f5eb-6ed7-4db1-944f-87406faa5a09/download/ttc-subway-delay-data-2021.xlsx"
}

# start dictionary with 2025 df
dfs_by_year = {2025: df_2025}

# Loop through each year and URL in xlsx_urls
for year, url in xlsx_urls.items():
    #User agent request
    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    
    #download excel into memory
    with urlopen(req) as resp:
        data = resp.read() # data bytes

    # read bytes into dfs
    df = pd.read_excel(BytesIO(data), engine="openpyxl")
    
    #add year key to each df
    df["year"] = year

    #add df to dictionary
    dfs_by_year[year] = df
    
#print headers for each to visually compare
for year, df in dfs_by_year.items():
    print(f"\n=== {year} ===")
    print(df.head())


=== 2025 ===
   _id        Date   Time        Day            Station   Code  Min Delay  \
0    1  2025-01-01  02:10  Wednesday   BATHURST STATION  MUSAN          5   
1    2  2025-01-01  02:30  Wednesday     DUNDAS STATION  MUIRS          0   
2    3  2025-01-01  02:32  Wednesday  BROADVIEW STATION  PUMST          0   
3    4  2025-01-01  02:58  Wednesday      KEELE STATION   EUSC          0   
4    5  2025-01-01  02:58  Wednesday    COXWELL STATION   SUAE          0   

   Min Gap Bound Line  Vehicle  
0        9     E   BD     5227  
1        0   NaN   YU        0  
2        0     E   BD        0  
3        0     W   BD     5293  
4        0   NaN   BD        0  

=== 2024 ===
        Date   Time     Day             Station   Code  Min Delay  Min Gap  \
0 2024-01-01  02:00  Monday    SHEPPARD STATION    MUI          0        0   
1 2024-01-01  02:00  Monday      DUNDAS STATION   MUIS          0        0   
2 2024-01-01  02:08  Monday      DUNDAS STATION  MUPAA          4       10   

In [89]:
# Not all dfs contain an id column.  Loop through and remove id. Only 2025 appears to have an id column, 
for year, df in dfs_by_year.items():
    if "_id" in df.columns:
        df.drop(columns="_id", inplace=True)

    # Ensure year column exists
    df["year"] = year

In [90]:
import re
# normalize all headers to lower and snake case 
for year, df in dfs_by_year.items():
    df.columns = [
        re.sub(r"[^0-9a-z_]", "", col.lower().replace(" ", "_"))
        for col in df.columns
    ]

In [91]:
#print headers for each to view changes
for year, df in dfs_by_year.items():
    print(f"\n=== {year} ===")
    print(df.head())


=== 2025 ===
         date   time        day            station   code  min_delay  min_gap  \
0  2025-01-01  02:10  Wednesday   BATHURST STATION  MUSAN          5        9   
1  2025-01-01  02:30  Wednesday     DUNDAS STATION  MUIRS          0        0   
2  2025-01-01  02:32  Wednesday  BROADVIEW STATION  PUMST          0        0   
3  2025-01-01  02:58  Wednesday      KEELE STATION   EUSC          0        0   
4  2025-01-01  02:58  Wednesday    COXWELL STATION   SUAE          0        0   

  bound line  vehicle  year  
0     E   BD     5227  2025  
1   NaN   YU        0  2025  
2     E   BD        0  2025  
3     W   BD     5293  2025  
4   NaN   BD        0  2025  

=== 2024 ===
        date   time     day             station   code  min_delay  min_gap  \
0 2024-01-01  02:00  Monday    SHEPPARD STATION    MUI          0        0   
1 2024-01-01  02:00  Monday      DUNDAS STATION   MUIS          0        0   
2 2024-01-01  02:08  Monday      DUNDAS STATION  MUPAA          4      

In [92]:
#Concatenate all files into one
df_subway_delays = pd.concat(
    dfs_by_year.values(),
    ignore_index=True
)



In [94]:
# Print the shape of the DataFrame
print(f"Shape of subway delays DataFrame: {df_subway_delays.shape} (rows, columns)")

# Print the column names
print("Columns in subway delays DataFrame:")
print(df_subway_delays.columns.tolist())

Shape of subway delays DataFrame: (98718, 11) (rows, columns)
Columns in subway delays DataFrame:
['date', 'time', 'day', 'station', 'code', 'min_delay', 'min_gap', 'bound', 'line', 'vehicle', 'year']


In [95]:
# Sort by station_name in ascending (A → Z) order
df_subway_delays = df_subway_delays.sort_values(by='station')


# Print the first 5 rows of df
print("First 5 rows of subway delays:")
print(df_subway_delays.head())

# Print the last 5 rows of df
print("\nLast 5 rows of subway delays:")
print(df_subway_delays.tail())

First 5 rows of subway delays:
                      date   time        day                 station   code  \
44412  2024-08-14 00:00:00  13:57  Wednesday   (APPROACHING) KENNEDY  PUSRA   
10145           2025-05-24  00:41   Saturday  1 ST CLAIR AVENUE EAST  MUIRS   
58546  2023-03-04 00:00:00  00:00   Saturday          1 TIPPETT ROAD  SUROB   
38345  2024-05-16 00:00:00  22:50   Thursday        111 SPADINA ROAD    SUO   
35489  2024-04-10 00:00:00  08:58  Wednesday  1900 YONGE MCBRIEN BLD   MUIE   

       min_delay  min_gap bound line  vehicle  year  
44412          4        8     E   BD     5341  2024  
10145          0        0   NaN   YU        0  2025  
58546          0        0   NaN  NaN        0  2023  
38345          0        0   NaN   YU        0  2024  
35489          0        0   NaN   YU        0  2024  

Last 5 rows of subway delays:
                      date   time        day                 station code  \
60989  2023-04-09 00:00:00  22:00     Sunday                  

In [96]:
# Normalize station names  formating Lowercase, strip spaces, remove 'station' if needed
df_subway_delays['station'] = (
    df_subway_delays['station']
    .str.lower()
    .str.strip()
    .str.replace(r'\bstation\b', '', regex=True)
)


In [75]:
# Sort by station_name in ascending (A → Z) order
df_subway_delays = df_subway_delays.sort_values(by='station')


# Print the first 5 rows of df
print("First 5 rows of subway delays:")
print(df_subway_delays.head())

# Print the last 5 rows of df
print("\nLast 5 rows of subway delays:")
print(df_subway_delays.tail())

First 5 rows of subway delays:
                      date   time        day                 station   code  \
93535  2022-10-20 00:00:00  06:34   Thursday                          MUATC   
44412  2024-08-14 00:00:00  13:57  Wednesday   (approaching) kennedy  PUSRA   
10145           2025-05-24  00:41   Saturday  1 st clair avenue east  MUIRS   
58546  2023-03-04 00:00:00  00:00   Saturday          1 tippett road  SUROB   
38345  2024-05-16 00:00:00  22:50   Thursday        111 spadina road    SUO   

       min_delay  min_gap bound line  vehicle  year  
93535          3        6     S   YU     5486  2022  
44412          4        8     E   BD     5341  2024  
10145          0        0   NaN   YU        0  2025  
58546          0        0   NaN  NaN        0  2023  
38345          0        0   NaN   YU        0  2024  

Last 5 rows of subway delays:
                      date   time        day                 station code  \
60989  2023-04-09 00:00:00  22:00     Sunday                  

In [ ]:
# Count the number of unique station names
num_unique_stations = df_subway_delays['station'].nunique()

print("Number of unique station names:", num_unique_stations)

# TTC Subway Station Names: Identifying Variations

In this section, I started to examine the `df_subway_delays` dataset to identify spelling inconsistencies and variations in TTC station names. The goals are to:

- **Compare** recorded station names against the official TTC station list.
- **Detect variations** such as directional indicators, platform information, or slight spelling differences.
- **Prepare** for standardizing station names for further analysis.

In [97]:
ttc_stations = [
    # Line 1 Yonge–University
    "vaughan metropolitan centre",
    "highway 407",
    "downsview park",
    "sheppard west",
    "wilson",
    "yorkdale",
    "lawrence west",
    "glencairn",
    "eglinton west",         # renamed cedarvale in 2025
    "st clair west",
    "dupont",
    "spadina",
    "st george",
    "museum",
    "queen's park",
    "st patrick",
    "osgoode",
    "st andrew",
    "union",
    "king",
    "queen",
    "dundas",
    "college",
    "wellesley",
    "bloor-yonge",
    "rosedale",
    "summerhill",
    "st clair",
    "davisville",
    "eglinton",
    "lawrence",
    "york mills",
    "sheppard-yonge",
    "north york centre",
    "finch",                
    
    # Line 2 Bloor–Danforth
    "kipling",
    "islington",
    "royal york",
    "old mill",
    "jane",
    "runnymede",
    "high park",
    "keele",
    "dundas west",
    "lansdowne",
    "dufferin",
    "ossington",
    "christie",
    "bathurst",
    "spadina",               # interchange w/Line 1
    "bay",
    "sherbourne",
    "castle frank",
    "broadview",
    "chester",
    "pape",
    "donlands",
    "greenwood",
    "coxwell",
    "woodbine",
    "main street",
    "victoria park",
    "warden",
    "kennedy",

    # Line 4 Sheppard
    "sheppard-yonge",        # interchange w/Line 1
    "bayview",
    "bessarion",
    "leslie",
    "don mills"
]

In [98]:
for ttc in ttc_stations:
    mask = df_subway_delays['station'].str.contains(ttc, na=False)
    matching_rows = df_subway_delays[mask][['station']].drop_duplicates()

    if not matching_rows.empty:
        print(f"\n=== Unique station names containing '{ttc}' ===")
        print(matching_rows)
        print(f"Total unique station names containing '{ttc}': {matching_rows.shape[0]}")


=== Unique station names containing 'highway 407' ===
                     station
91350           highway 407 
29776        highway 407  to
61404  union and highway 407
Total unique station names containing 'highway 407': 3

=== Unique station names containing 'downsview park' ===
               station
94502  downsview park 
Total unique station names containing 'downsview park': 1

=== Unique station names containing 'sheppard west' ===
                      station
41079      2233 sheppard west
43046  ea sheppard west to st
18714   sheppard west - finch
71381  sheppard west - st cla
69557  sheppard west ee (3940
66520    sheppard west sation
85423          sheppard west 
37177  sheppard west to finch
43124  sheppard west to st cl
78599  sheppard west to union
90247  sheppard west to wilso
89184  union to sheppard west
Total unique station names containing 'sheppard west': 12

=== Unique station names containing 'wilson' ===
                      station
44371         finch to wils